In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from neuralop.losses.differentiation import FiniteDiff
import os
import scipy.io
from neuralop.models import FNO
import torch.nn.functional as F 

device = torch.device("cuda") # Use GPU 

In [2]:
root_dir = '/home/seongwon/AI/neural operator/data/Burgers1dTimeDataset'
os.makedirs(root_dir, exist_ok=True)

# resolution 256

In [3]:
train_data_256 = torch.load(os.path.join(root_dir, 'burgers_train_256.pt'), map_location=device)
test_data_256  = torch.load(os.path.join(root_dir, 'burgers_test_256.pt'),  map_location=device)

x_train_256 = train_data_256['x']  # [1000, 256]
x_train_256 = x_train_256.unsqueeze(1)  # [1000, 1, 256]
y_train_256 = train_data_256['y']  # [1000, 256]
y_train_256 = y_train_256.unsqueeze(1)  # [1000, 1, 256]
x_test_256  = test_data_256['x']   # [200, 256]
x_test_256  = x_test_256.unsqueeze(1)   # [200, 1, 256]
y_test_256  = test_data_256['y']   # [200, 256]
y_test_256  = y_test_256.unsqueeze(1)   # [200, 1, 256]

print(f"train x: {x_train_256.shape}, y: {y_train_256.shape}")
print(f"test  x: {x_test_256.shape},  y: {y_test_256.shape}")

train x: torch.Size([1000, 1, 256]), y: torch.Size([1000, 1, 256])
test  x: torch.Size([200, 1, 256]),  y: torch.Size([200, 1, 256])


In [4]:
model = FNO(n_modes=(16,),
            hidden_channels=64,
            in_channels=1,
            out_channels=1,
            factorization='tucker',
            non_linearity= F.relu
            ).to(device)

In [3]:
# 데이터가 무거우니 interative하게 만듦
from torch.utils.data import DataLoader, TensorDataset

train_dataset_256 = TensorDataset(x_train_256, y_train_256)
test_dataset_256 = TensorDataset(x_test_256, y_test_256)

train_loader_256 = DataLoader(train_dataset_256, batch_size=40, shuffle=False)
test_loader_256 = DataLoader(test_dataset_256, batch_size=40, shuffle=False)

NameError: name 'x_train_256' is not defined

In [4]:
def relative_l2_loss(pred, target):
    # 샘플별로 계산 후 평균
    diff = torch.norm(pred - target, dim=-1)
    norm = torch.norm(target, dim=-1)
    return torch.mean(diff / norm)

In [7]:
#학습 시키기 위한 준비

import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5) # learning rate is halved every 100 epochs

In [8]:
device = torch.device('cuda')
model.to(device)

epochs = 500
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x, y in train_loader_256:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        output = model(x)
        loss = relative_l2_loss(output, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    scheduler.step()
    if epoch % 1 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader_256):.6f}")

Epoch [1/500], Loss: 0.788988
Epoch [2/500], Loss: 0.584504
Epoch [3/500], Loss: 0.496205
Epoch [4/500], Loss: 0.455488
Epoch [5/500], Loss: 0.401369
Epoch [6/500], Loss: 0.360350
Epoch [7/500], Loss: 0.305312
Epoch [8/500], Loss: 0.327162
Epoch [9/500], Loss: 0.248739
Epoch [10/500], Loss: 0.223989
Epoch [11/500], Loss: 0.290312
Epoch [12/500], Loss: 0.233085
Epoch [13/500], Loss: 0.259544
Epoch [14/500], Loss: 0.249350
Epoch [15/500], Loss: 0.194744
Epoch [16/500], Loss: 0.165170
Epoch [17/500], Loss: 0.242093
Epoch [18/500], Loss: 0.181350
Epoch [19/500], Loss: 0.148644
Epoch [20/500], Loss: 0.216558
Epoch [21/500], Loss: 0.167316
Epoch [22/500], Loss: 0.144964
Epoch [23/500], Loss: 0.162435
Epoch [24/500], Loss: 0.228666
Epoch [25/500], Loss: 0.240230
Epoch [26/500], Loss: 0.215132
Epoch [27/500], Loss: 0.176632
Epoch [28/500], Loss: 0.139279
Epoch [29/500], Loss: 0.202628
Epoch [30/500], Loss: 0.182643
Epoch [31/500], Loss: 0.154449
Epoch [32/500], Loss: 0.129688
Epoch [33/500], L

In [9]:
# 평가
model.eval()
total_test_loss = 0  # 루프 밖으로

for x, y in test_loader_256:
    x, y = x.to(device), y.to(device)
    with torch.no_grad():
        output = model(x)
        test_loss = relative_l2_loss(output, y)
        total_test_loss += test_loss.item()
    print(f"Batch Test Loss: {test_loss.item():.6f}")

print(f"Average Test Loss: {total_test_loss / len(test_loader_256):.6f}")

Batch Test Loss: 0.171305
Batch Test Loss: 0.161689
Batch Test Loss: 0.170958
Batch Test Loss: 0.168455
Batch Test Loss: 0.184530
Average Test Loss: 0.171387


In [10]:
np.log10(total_test_loss / len(test_loader_256))

np.float64(-0.7660209327588471)

# resolution 2048

In [5]:
train_data_2048 = torch.load(os.path.join(root_dir, 'burgers_train_2048.pt'), map_location=device)
test_data_2048  = torch.load(os.path.join(root_dir, 'burgers_test_2048.pt'),  map_location=device)

x_train_2048 = train_data_2048['x']  # [1000, 2048]
x_train_2048 = x_train_2048.unsqueeze(1)  # [1000, 1, 2048]
y_train_2048 = train_data_2048['y']  # [1000, 2048]
y_train_2048 = y_train_2048.unsqueeze(1)  # [1000, 1, 2048]
x_test_2048  = test_data_2048['x']   # [200, 2048]
x_test_2048  = x_test_2048.unsqueeze(1)   # [200, 1, 2048]
y_test_2048  = test_data_2048['y']   # [200, 2048]
y_test_2048  = y_test_2048.unsqueeze(1)   # [200, 1, 2048]

print(f"train x: {x_train_2048.shape}, y: {y_train_2048.shape}")
print(f"test  x: {x_test_2048.shape},  y: {y_test_2048.shape}")

train x: torch.Size([1000, 1, 2048]), y: torch.Size([1000, 1, 2048])
test  x: torch.Size([200, 1, 2048]),  y: torch.Size([200, 1, 2048])


In [6]:
model_2048 = FNO(n_modes=(16,),
            hidden_channels=64,
            in_channels=1,
            out_channels=1,
            factorization='tucker',
            non_linearity= F.relu
            ).to(device)

In [7]:
# 데이터가 무거우니 interative하게 만듦
from torch.utils.data import DataLoader, TensorDataset

train_dataset_2048 = TensorDataset(x_train_2048, y_train_2048)
test_dataset_2048 = TensorDataset(x_test_2048, y_test_2048)

train_loader_2048 = DataLoader(train_dataset_2048, batch_size=40, shuffle=False)
test_loader_2048 = DataLoader(test_dataset_2048, batch_size=40, shuffle=False)

In [9]:
#학습 시키기 위한 준비

import torch.optim as optim

optimizer = optim.Adam(model_2048.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5) # learning rate is halved every 100 epochs

In [11]:
device = torch.device('cuda')
model_2048.to(device)

epochs = 500
for epoch in range(epochs):
    model_2048.train()
    total_loss = 0
    for x, y in train_loader_2048:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        output = model_2048(x)
        loss = relative_l2_loss(output, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    scheduler.step()
    if epoch % 1 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader_2048):.6f}")

Epoch [1/500], Loss: nan
Epoch [2/500], Loss: nan
Epoch [3/500], Loss: nan
Epoch [4/500], Loss: nan
Epoch [5/500], Loss: nan


KeyboardInterrupt: 

In [21]:
# 평가
model_2048.eval()
total_test_loss = 0  # 루프 밖으로

for x, y in test_loader_2048:
    x, y = x.to(device), y.to(device)
    with torch.no_grad():
        output = model_2048(x)
        test_loss = relative_l2_loss(output, y)
        total_test_loss += test_loss.item()
    print(f"Batch Test Loss: {test_loss.item():.6f}")

print(f"Average Test Loss: {total_test_loss / len(test_loader_2048):.6f}")

Batch Test Loss: nan
Batch Test Loss: nan
Batch Test Loss: nan
Batch Test Loss: nan
Batch Test Loss: nan
Average Test Loss: nan


In [22]:
np.log10(total_test_loss / len(test_loader_256))

np.float64(nan)